# 0.0 Usage
Follow the steps below:
1. Upload the 3 data files (hero_info.csv, tournament_data.csv, consoliadted_game_info.csv) into the "folder" icon on the left hand side.
2. Set your filtration rules in section 1, i.e. which tournaments and stages you want to include in the analysis
3. Run the remaining section in sequence.
4. Check the result in .csv format.

# 1.0 Set Filtration Rules
Set your filtration rules by changing the 5 variables below. Only tournaments/games that fullfill ALL condition will be kept for analysis purpose.
*   tournament_codes: List of tournament codes you want to include in the analysis (refer to tournament_data.csv for the code mapping).
*   tournament_tiers: List of tournament tiers (S, A, B) to include.
*   tournament_stages: 'b' for bracket (ie knockout stage) only.
*   tournament_start_date: The earliest start date for tournaments (inclusive). The format should be yyyymmdd, eg 7 April 2020 should be 20200407
*   tournament_end_date: The latest end date for tournaments (inclusive). Format same as above.
<br>

For no filtration, put as None.

--------- Example 1: \
- tournament_codes = [1] \
- tournament_tiers = None \
- tournament_stages = 'b' \
- tournament_start_date = None \
- tournament_end_date = None \
<br>

Includes only M5, any tier tournament, at bracket stage, at any time period. \
Basically means only M5 bracket stage data.
<br>

--------- Example 2: \
- tournament_codes = None \
- tournament_tiers = ['S', 'A'] \
- tournament_stages = None \
- tournament_start_date = 20230101 \
- tournament_end_date = 20231231 \
<br>

Includes all tournaments, of S or A tier, at all tournament stages, that is within 1 Jan 2023 to 31 Dec 2023. \
Basically means S and A tiers tournament that starts and end in year 2023. \

In [24]:
tournament_codes = [1]
tournament_tiers = None
tournament_stages = 'b'
tournament_start_date = 20230101
tournament_end_date = 20231231

# 2.0 Run Preparation Codes
Run everything in this section if you are new to coding. Just click the "play" triangular button to run.

## 2.1 Import Library and Utils Functions

In [25]:
import os
import pandas as pd
from datetime import datetime
from tqdm import tqdm
from google.colab import files

## 2.2 Utils Functions

In [26]:
def _adjust_hero_name(hero_name):

    # if 'popol' in hero_name.lower():
    #     return 'popol'

    return hero_name.lower()

def replace_error_date(date_str):
    if len(str(date_str)) != 8:
        return None
    else:
        return date_str

def robust_division(numerator, denom, error_value=0):
    if denom == 0:
        return error_value
    else:
        return numerator/denom

def convert_bp_str_to_list(bp_in_str):
    if pd.isna(bp_in_str):
        return []

    if bp_in_str[0] == '(' and bp_in_str[-1] == ')':
        inner_str = bp_in_str[1:-1]
        if inner_str == '':
            return []
        else:
            inner_list = inner_str.split(',')
            return [name.strip("' ").lower() for name in inner_list]
    else:
        return bp_in_str


def check_ban_pick_result(hero_name, t1_name, t1_side, t1_bans, t1_picks, t1_result, t2_name, t2_side, t2_bans, t2_picks, t2_result):

    if hero_name in t1_bans:
        ban_side = t1_side
        ban_against_team = t2_name
    elif hero_name in t2_bans:
        ban_side = t2_side
        ban_against_team = t1_name
    else:
        ban_side = None
        ban_against_team = None

    if hero_name in t1_picks:
        pick_side = t1_side
        pick_team = t1_name
        result = t1_result
    elif hero_name in t2_picks:
        pick_side = t2_side
        pick_team = t2_name
        result = t2_result
    else:
        pick_side = None
        pick_team = None
        result = None

    return ban_side, ban_against_team, pick_side, pick_team, result

def get_sides_bp_stats(bp_side_df):
    ban_counts = bp_side_df['ban_side'].value_counts()
    pick_counts = bp_side_df['pick_side'].value_counts()

    return {
        'blue_ban_num': ban_counts.get('blue', 0),
        'blue_pick_num': pick_counts.get('blue', 0),
        'red_ban_num': ban_counts.get('red', 0),
        'red_pick_num': pick_counts.get('red', 0),
    }

def get_win_lose_stats(res_gametime_df):
    win_games_df = res_gametime_df[res_gametime_df['result'] == 1]
    win_num = win_games_df.shape[0]
    win_avg_game_time_sec = win_games_df['game_time_sec'].mean()
    if pd.isna(win_avg_game_time_sec):
        win_avg_game_time_sec = 0

    lose_games_df = res_gametime_df[res_gametime_df['result'] == 0]
    lose_num = lose_games_df.shape[0]
    lose_avg_game_time_sec = lose_games_df['game_time_sec'].mean()
    if pd.isna(lose_avg_game_time_sec):
        lose_avg_game_time_sec = 0


    return {
        'full_win_num': win_num,
        'full_lose_num': lose_num,
        'full_win_avg_game_time_sec': round(win_avg_game_time_sec),
        'full_lose_avg_game_time_sec': round(lose_avg_game_time_sec),
    }

def get_derived_stats(num_games, sides_bp_stats, win_lose_stats):
    full_ban_num = sides_bp_stats['red_ban_num'] + sides_bp_stats['blue_ban_num']
    full_ban_rate = robust_division(full_ban_num, num_games, 0)
    full_pick_num = sides_bp_stats['red_pick_num'] + sides_bp_stats['blue_pick_num']
    full_pick_rate = robust_division(full_pick_num, num_games-full_ban_num, 0)
    full_bp_num = full_ban_num + full_pick_num
    full_bp_rate = robust_division(full_bp_num, num_games, 0)
    #print(full_pick_num == (win_lose_stats['full_win_num'] + win_lose_stats['full_lose_num']))
    full_win_rate = robust_division(win_lose_stats['full_win_num'], full_pick_num, 0)

    blue_ban_ratio = robust_division(sides_bp_stats['blue_ban_num'], full_ban_num, 0)
    red_ban_ratio = robust_division(sides_bp_stats['red_ban_num'], full_ban_num, 0)
    blue_pick_rate = robust_division(sides_bp_stats['blue_pick_num'], num_games - full_ban_num - sides_bp_stats['red_pick_num'], 0)
    red_pick_rate = robust_division(sides_bp_stats['red_pick_num'], num_games - full_ban_num - sides_bp_stats['blue_pick_num'], 0)

    return {
        'blue_ban_ratio': round(blue_ban_ratio, 4),
        'red_ban_ratio': round(red_ban_ratio, 4),
        'blue_pick_rate': round(blue_pick_rate, 4),
        'red_pick_rate': round(red_pick_rate, 4),
        'full_ban_num': full_ban_num,
        'full_ban_rate': round(full_ban_rate, 4),
        'full_pick_num': full_pick_num,
        'full_pick_rate': round(full_pick_rate, 4),
        'full_bp_num': full_bp_num,
        'full_bp_rate': round(full_bp_rate, 4),
        'full_win_rate': round(full_win_rate, 4)
    }

def get_notable_teams_stats(expanded_game_data_df):
    # TODO: get pick rate for each team, then get most pick rate team

    pick_games_df = expanded_game_data_df[expanded_game_data_df['pick_side'].notna()]
    pick_team_summary = pick_games_df.groupby('pick_team').agg(
        pick_num = ('pick_team', 'size'),
        win_num = ('result', 'sum')
    )
    if pick_team_summary.empty:
        pick_notable_teams_dict = {
            'mpnt_team_name': "",
            'mpnt_pick_num': 0,
            'mpnt_win_num': 0,
            'mpnt_win_rate': 0,
            'mwrt_team_name': "",
            'mwrt_pick_num': 0,
            'mwrt_win_num': 0,
            'mwrt_win_rate': 0,
        }
    else:
        pick_team_summary.reset_index(inplace=True)
        pick_team_summary['win_rate'] = pick_team_summary['win_num'] / pick_team_summary['pick_num']

        # get most pick number team
        mpnt = pick_team_summary.sort_values(by = ['pick_num', 'win_rate'], ascending=False).head(1)

        # get most win rate team
        mwrt = pick_team_summary.sort_values(by = ['win_rate', 'pick_num'], ascending=False).head(1)

        pick_notable_teams_dict = {
            'mpnt_team_name': mpnt['pick_team'].iloc[0],
            'mpnt_pick_num': mpnt['pick_num'].iloc[0],
            'mpnt_win_num': mpnt['win_num'].iloc[0],
            'mpnt_win_rate': round(mpnt['win_rate'].iloc[0], 4),
            'mwrt_team_name': mwrt['pick_team'].iloc[0],
            'mwrt_pick_num': mwrt['pick_num'].iloc[0],
            'mwrt_win_num': mwrt['win_num'].iloc[0],
            'mwrt_win_rate': round(mwrt['win_rate'].iloc[0], 4),
        }



    ban_games_df = expanded_game_data_df[expanded_game_data_df['ban_side'].notna()]
    ban_team_summary = ban_games_df.groupby('ban_against_team').agg(
        ban_num = ('ban_against_team', 'size'),
    )
    if ban_team_summary.empty:
        ban_notable_teams_dict =  {
            'mbat_team_name': "",
            'mbat_ban_num': 0,
        }
    else:
        ban_team_summary.reset_index(inplace=True)

        # get most ban against team
        mbat = ban_team_summary.sort_values(by = ['ban_num'], ascending=False).head(1)

        ban_notable_teams_dict = {
            'mbat_team_name': mbat['ban_against_team'].iloc[0],
            'mbat_ban_num':  mbat['ban_num'].iloc[0],
        }

    notable_teams_dict = {}
    notable_teams_dict.update(pick_notable_teams_dict)
    notable_teams_dict.update(ban_notable_teams_dict)

    return notable_teams_dict

def get_all_stats(num_games, expanded_game_data_df):
    sides_bp_stats = get_sides_bp_stats(expanded_game_data_df[['ban_side', 'pick_side']])
    win_lose_stats = get_win_lose_stats(expanded_game_data_df[['result', 'game_time_sec']])
    derived_stats = get_derived_stats(num_games, sides_bp_stats, win_lose_stats)
    notable_team_stats = get_notable_teams_stats(expanded_game_data_df)

    mbat_ban_ratio = robust_division(notable_team_stats['mbat_ban_num'], derived_stats['full_ban_num'], 0)

    all_stats = {
        'mbat_ban_ratio': mbat_ban_ratio
    }
    all_stats.update(sides_bp_stats)
    all_stats.update(win_lose_stats)
    all_stats.update(derived_stats)
    all_stats.update(notable_team_stats)

    return all_stats

## 2.3 Read datasets

In [27]:
# dataset names
tournament_data_name = "tournament_data.csv"
game_data_name = "consolidated_game_data.csv"
hero_info_name = "hero_info.csv"

# read hero info
hero_info_df = pd.read_csv(hero_info_name, usecols=['Name', 'Hero Code'])
hero_info_df['Name'] = hero_info_df['Name'].apply(_adjust_hero_name)

# read tournament data
tournament_data_df = pd.read_csv(tournament_data_name, dtype=str)
# replace errornous date
tournament_data_df['start_date'] = tournament_data_df['start_date'].apply(replace_error_date)
tournament_data_df['start_date'] = pd.to_datetime(tournament_data_df['start_date'], format='%Y%m%d')
tournament_data_df['end_date'] = tournament_data_df['end_date'].apply(replace_error_date)
tournament_data_df['end_date'] = pd.to_datetime(tournament_data_df['end_date'], format='%Y%m%d')

# read game data
game_data_df = pd.read_csv(game_data_name, dtype = {
    'tournament_code': str,
    'date': str,
    'game_time_str': str
})
game_data_df['t1_picks'] = game_data_df['t1_picks'].apply(convert_bp_str_to_list)
game_data_df['t1_bans'] = game_data_df['t1_bans'].apply(convert_bp_str_to_list)
game_data_df['t2_picks'] = game_data_df['t2_picks'].apply(convert_bp_str_to_list)
game_data_df['t2_bans'] = game_data_df['t2_bans'].apply(convert_bp_str_to_list)

## 2.4 Filter Game Data

In [28]:
no_game_data = False
## tournament level
filter_condition_list = []

if tournament_codes is not None:
    tournament_codes_str = [str(code) for code in tournament_codes]
    filter_condition_list.append(tournament_data_df['tournament_code'].isin(tournament_codes_str))

if tournament_tiers is not None:
    filter_condition_list.append(tournament_data_df['tier'].isin(tournament_tiers))

if tournament_start_date is not None:
    start_date_dt = pd.to_datetime(tournament_start_date, format='%Y%m%d')
    filter_condition_list.append(tournament_data_df['start_date'] >= start_date_dt)

if tournament_end_date is not None:
    end_date_dt = pd.to_datetime(tournament_end_date, format='%Y%m%d')
    filter_condition_list.append(tournament_data_df['end_date'] <= end_date_dt)

combined_condition = pd.Series([True] * len(tournament_data_df))

for condition in filter_condition_list:
    combined_condition = combined_condition & condition

filtered_tournament_df = tournament_data_df[combined_condition]


## game data level
updated_tournament_codes = filtered_tournament_df['tournament_code'].tolist()
tournament_codes_condition = game_data_df['tournament_code'].isin(updated_tournament_codes)

if tournament_stages == 'b':
    tournament_stages_condition = game_data_df['tournament_stage'] == 'bracket'
else:
    tournament_stages_condition = pd.Series([True] * len(game_data_df))

filtered_game_data_df = game_data_df[tournament_codes_condition & tournament_stages_condition]

# check if there is no game data
if filtered_game_data_df.shape[0] == 0:
  no_game_data = True

# 3.0 Do Analysis
If your filtration rule leads to no game at all, it should show: \
"Filtration rule leads to no game data, please reset them." \
<br>
Else, it will show a progress bar, then a final line like: \\
"Analysis results saved as '20231230_0000_hero_bpw_table.csv'"


In [29]:
if no_game_data is True:
  print("Filtration rule leads to no game data, please reset them.")
else:
  num_games = filtered_game_data_df.shape[0]

  stats_dict_list = []

  # loop through all heroes
  hero_rows_pbar = tqdm(hero_info_df.iterrows(), total = hero_info_df.shape[0], desc = "Progress")
  for _, row in hero_rows_pbar:
      hero_code = row['Hero Code']
      hero_name = row['Name']

      hero_rows_pbar.set_postfix({
          'Hero': hero_name
      })

      # create a new copy of game data df for analysis
      temp_game_data_df = filtered_game_data_df.copy(deep=True)
      temp_game_data_df[['ban_side', 'ban_against_team', 'pick_side', 'pick_team', 'result']] = temp_game_data_df.apply(
          lambda row: check_ban_pick_result(hero_name,
              row['t1_name'], row['t1_side'], row['t1_bans'], row['t1_picks'], row['t1_result'],
              row['t2_name'], row['t2_side'], row['t2_bans'], row['t2_picks'], row['t2_result'],
          ),
          axis=1, result_type='expand'
      )

      #temp_game_data_df.to_csv(os.path.join(result_dir, 'test.csv'))
      all_stats = get_all_stats(num_games, temp_game_data_df)
      final_dict = {
          "hero_name": hero_name,
          "num_games": num_games
      }
      final_dict.update(all_stats)

      stats_dict_list.append(final_dict)


  # aggregate all dict into 1, then convert to df
  full_stats_dict = {}
  for key in stats_dict_list[0].keys():
      full_stats_dict[key] = [d[key] for d in stats_dict_list]

  full_stats_df = pd.DataFrame(full_stats_dict)
  full_stats_df.sort_values(by = ['full_bp_rate', 'full_ban_rate', 'full_win_rate'], ascending=False, inplace=True)
  full_stats_df = full_stats_df[[
      'hero_name', 'num_games',
      'blue_ban_num', 'red_ban_num', 'full_ban_num', 'full_ban_rate', 'blue_ban_ratio', 'red_ban_ratio',
      'mbat_team_name', 'mbat_ban_num', 'mbat_ban_ratio',
      'blue_pick_num', 'blue_pick_rate',
      'red_pick_num', 'red_pick_rate',
      'full_pick_num', 'full_pick_rate',
      'mpnt_team_name', 'mpnt_pick_num', 'mpnt_win_num', 'mpnt_win_rate',
      'full_bp_num', 'full_bp_rate',
      'full_win_num', 'full_lose_num', 'full_win_rate',
      'full_win_avg_game_time_sec', 'full_lose_avg_game_time_sec',
      'mwrt_team_name', 'mwrt_pick_num', 'mwrt_win_num', 'mwrt_win_rate',
  ]]



  # output analysis result
  curr_dt = datetime.now().strftime("%Y%m%d_%H%M%S")
  analysis_result_name = f"{curr_dt}_hero_bpw_table.csv"

  full_stats_df.to_csv(analysis_result_name, index=False)
  print(f"\nAnalysis results saved as '{analysis_result_name}'")

Progress: 100%|██████████| 123/123 [00:03<00:00, 34.94it/s, Hero=cici]



Analysis results saved as '20231230_060451_hero_bpw_table.csv'


# 4.0 Download the Result File
You can get the analysis result file from left hand side, or simply run the code below to download it. \
To repeat analysis with different filtration rules, just rerun everything from Section 1.0.

In [30]:
files.download(analysis_result_name)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>